# 04 · Entrenamiento final, evaluación en test y entrenamiento distribuido

**Temas 4 y 6 del proyecto.**

1. **Entrenamiento distribuido:** se mide cuánto se acelera el entrenamiento al repartir cada
   lote entre las 2 GPU T4 (`MirroredStrategy`) frente a una sola GPU.
2. **Entrenamiento final de los 5 modelos** (3 CNN y 2 transformers) con el train completo
   (52.292 imágenes), los hiperparámetros de Hyperband y las 2 GPU.
3. **Evaluación en test**, que se usa aquí por primera y única vez: exactitud, F1, matriz de
   confusión, tiempo de inferencia y tamaño de cada modelo.

**Cada entrenamiento corre en un proceso de Python aparte** (`entrenar.py`). Varios
`MirroredStrategy` seguidos dentro del mismo proceso terminan chocando entre sí: las dos GPU
quedan sincronizando modelos distintos y TensorFlow aborta la comunicación entre ellas. Un
proceso nuevo por modelo arranca siempre limpio. Si un modelo falla con 2 GPU, se reintenta
automáticamente con 1.

**Requisitos de la sesión:** GPU T4 x2, Internet On y, como Input, el dataset y las salidas de
`01-preparacion-datos` y `02_hyperband`.

**Salida:** `modelo_<nombre>.keras` (los 5 modelos entrenados, que usa la aplicación web),
`val_probs_<nombre>.npy` y `test_probs_<nombre>.npy` (para los ensambles del tema 5),
`history_<nombre>.csv`, `test_resultados.csv`, `test_f1_por_clase.csv` y `distribuido.csv`.

### Modo de prueba

Con `QUICK = True` todo corre en miniatura para detectar errores en pocos minutos.
**Para la corrida real ponlo en `False` antes de Save & Run All.**

In [ ]:
QUICK = True

In [ ]:
# Este notebook no importa TensorFlow: así el proceso principal no ocupa memoria de las GPU,
# que quedan libres para los procesos de entrenamiento.
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             f1_score, log_loss, precision_score, recall_score)

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
MODEL_NAMES = ["cnn_propia", "efficientnetb0", "resnet50v2", "vit_b16", "deit_s"]


def find_dir_with(*names, base="/kaggle/input"):
    for root, _, files in os.walk(base):
        if all(n in files for n in names):
            return Path(root)
    raise FileNotFoundError(f"No se encontró {names}: revisa los Input del notebook")


SPLITS = find_dir_with("train.csv", "rice_data.py")
HB = find_dir_with("best_hparams.csv", "rice_models.py")
CLASSES = (pd.read_csv(SPLITS / "test.csv")[["label_idx", "label"]]
           .drop_duplicates().sort_values("label_idx")["label"].tolist())
print("Particiones:", SPLITS)
print("Hyperband:", HB)
print("Clases:", CLASSES)

## 1. Script de entrenamiento `entrenar.py`

Entrena **un** modelo en un proceso propio y guarda todo en disco: el modelo, el historial de
épocas, las probabilidades en validación y test, y un JSON con tiempos y tamaño. Con
`--bench N` solo cronometra una época con N imágenes (para la comparación 1 GPU / 2 GPU).

In [ ]:
%%writefile entrenar.py
"""Entrena un modelo del proyecto en su propio proceso (1 o 2 GPU) y guarda los resultados."""
import argparse
import json
import os
import sys
import time
from pathlib import Path

os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

p = argparse.ArgumentParser()
p.add_argument("--model", required=True)
p.add_argument("--gpus", type=int, default=2)
p.add_argument("--quick", action="store_true")
p.add_argument("--bench", type=int, default=0)
p.add_argument("--splits", required=True)
p.add_argument("--hb", required=True)
p.add_argument("--out", required=True)
a = p.parse_args()
sys.path += [a.splits, a.hb]

import keras
import numpy as np
import pandas as pd
import tensorflow as tf

from rice_data import SEED, find_data_root, make_dataset
from rice_models import build_model

keras.mixed_precision.set_global_policy("mixed_float16")
keras.utils.set_random_seed(SEED)
OUT = Path(a.out)
DATA_ROOT = find_data_root()

if a.gpus > 1:
    strategy = tf.distribute.MirroredStrategy()
else:
    strategy = tf.distribute.OneDeviceStrategy("/gpu:0")
R = strategy.num_replicas_in_sync

train_df = pd.read_csv(f"{a.splits}/train.csv")
val_df = pd.read_csv(f"{a.splits}/val.csv")
test_df = pd.read_csv(f"{a.splits}/test.csv")
if a.quick:
    muestra = lambda df, n: df.groupby("label_idx", group_keys=False).sample(n=n, random_state=SEED)
    train_df, val_df, test_df = muestra(train_df, 60), muestra(val_df, 20), muestra(test_df, 20)
train_df, val_df, test_df = (d.reset_index(drop=True) for d in (train_df, val_df, test_df))

if a.bench:
    # Una época cronometrada con el mismo lote por GPU (32), después de compilar.
    df = train_df.groupby("label_idx", group_keys=False).sample(n=a.bench // 5, random_state=SEED)
    with strategy.scope():
        model = build_model(a.model, lr=1e-4)
    ds = make_dataset(df, DATA_ROOT, batch_size=32 * R, training=True)
    model.fit(ds.take(2), epochs=1, verbose=0)
    t0 = time.time()
    model.fit(ds, epochs=1, verbose=0)
    print("RESULTADO " + json.dumps({"gpus": R, "imagenes": len(df), "segundos": time.time() - t0}))
    sys.exit(0)

hp = pd.read_csv(f"{a.hb}/best_hparams.csv").set_index("model").loc[a.model]
# Con 5 veces más datos por época que en Hyperband, se limita a 5 épocas; early stopping corta antes.
epochs = 1 if a.quick else int(min(5, 5 if pd.isna(hp["epochs"]) else hp["epochs"]))
print(f"{a.model}: {R} GPU, lr={hp['lr']:.2e}, dropout={hp['dropout']}, "
      f"lote por GPU={int(hp['batch_size'])}, épocas máx.={epochs}", flush=True)

with strategy.scope():
    model = build_model(a.model, lr=float(hp["lr"]), dropout=float(hp["dropout"]))
val_ds = make_dataset(val_df, DATA_ROOT, batch_size=64)
test_ds = make_dataset(test_df, DATA_ROOT, batch_size=64)

es = keras.callbacks.EarlyStopping("val_loss", patience=2, restore_best_weights=True)
t0 = time.time()
hist = model.fit(
    make_dataset(train_df, DATA_ROOT, batch_size=int(hp["batch_size"]) * R, training=True),
    validation_data=val_ds,
    epochs=epochs,
    callbacks=[es, keras.callbacks.TerminateOnNaN()],
    verbose=2,
)
t_train = time.time() - t0

# Si la pérdida llegó a NaN, el modelo no sirve: se sale con error para que el notebook reintente.
if any(np.isnan(v) for v in hist.history["loss"]):
    print(f"ERROR: la pérdida de {a.model} llegó a NaN con {R} GPU", flush=True)
    sys.exit(3)
# Se fijan explícitamente los pesos de la mejor época según val_loss.
if es.best_weights is not None:
    model.set_weights(es.best_weights)

ruta = OUT / f"modelo_{a.model}.keras"
model.save(ruta)
pd.DataFrame(hist.history).to_csv(OUT / f"history_{a.model}.csv", index_label="epoch")
np.save(OUT / f"val_probs_{a.model}.npy", np.asarray(model.predict(val_ds, verbose=0), dtype="float32"))
np.save(OUT / f"test_probs_{a.model}.npy", np.asarray(model.predict(test_ds, verbose=0), dtype="float32"))
np.save(OUT / "val_y.npy", val_df["label_idx"].to_numpy())
np.save(OUT / "test_y.npy", test_df["label_idx"].to_numpy())

# Tiempo de inferencia por imagen, ya compilado (sin contar la primera llamada).
lote = next(iter(make_dataset(test_df.head(64), DATA_ROOT, batch_size=64)))[0]
model.predict(lote, verbose=0)
t0 = time.time()
for _ in range(5):
    model.predict(lote, verbose=0)
ms_img = (time.time() - t0) / (5 * len(lote)) * 1000

info = {"model": a.model, "gpus": R, "epochs_run": len(hist.history["loss"]),
        "train_min": t_train / 60, "infer_ms_img": ms_img,
        "params_M": model.count_params() / 1e6, "size_MB": ruta.stat().st_size / 1e6}
(OUT / f"info_{a.model}.json").write_text(json.dumps(info))
print("RESULTADO " + json.dumps(info))

In [ ]:
def correr(*args):
    """Ejecuta entrenar.py en un proceso nuevo, muestra su salida y devuelve (código, resultado)."""
    cmd = [sys.executable, "entrenar.py", "--splits", str(SPLITS), "--hb", str(HB),
           "--out", str(OUT_DIR), *args] + (["--quick"] if QUICK else [])
    env = {**os.environ, "TF_CPP_MIN_LOG_LEVEL": "2", "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    resultado = None
    for linea in proc.stdout:
        if "cuda_timer" in linea or linea.startswith(("I0000", "W0000")):
            continue
        if linea.startswith("RESULTADO "):
            resultado = json.loads(linea[len("RESULTADO "):])
        print(linea, end="", flush=True)
    return proc.wait(), resultado

## 2. Entrenamiento distribuido: 1 GPU frente a 2 GPU

**Paralelismo de datos con `MirroredStrategy`:** cada GPU guarda una copia completa del modelo
y procesa la mitad de cada lote. Al final de cada paso, los gradientes de las dos copias se
promedian (*all-reduce*) para que ambas queden con los mismos pesos.

Se entrena una época de DeiT-S con cada estrategia, con el mismo lote por GPU (32), y se
descarta el tiempo de compilación. Se usa un transformer porque, como muestra la sección 3, las
CNN no entrenan bien con `MirroredStrategy` en esta configuración. En modo de prueba hay tan pocas imágenes que el resultado no
significa nada; la medición válida es la de la corrida real (3.000 imágenes).

In [ ]:
N_BENCH = 250 if QUICK else 3000
filas_d = []
for g in (1, 2):
    codigo, r = correr("--model", "deit_s", "--gpus", str(g), "--bench", str(N_BENCH))
    if codigo == 0 and r:
        filas_d.append({"GPU": r["gpus"], "segundos": r["segundos"], "imágenes/s": r["imagenes"] / r["segundos"]})
    else:
        print(f"La medición con {g} GPU falló (código {codigo})")

distribuido = pd.DataFrame(filas_d)
if len(distribuido) == 2:
    distribuido["aceleración"] = distribuido["imágenes/s"] / distribuido.loc[0, "imágenes/s"]
distribuido.to_csv(OUT_DIR / "distribuido.csv", index=False)
distribuido.round(2)

La aceleración ideal con 2 GPU sería 2×. En la práctica es menor porque las GPU tienen que
sincronizar gradientes en cada paso y porque la lectura de imágenes desde disco (CPU) puede
volverse el cuello de botella.

## 3. Entrenamiento final de los 5 modelos

**Hallazgo de la primera corrida:** con `MirroredStrategy` las tres CNN no aprendieron. La CNN
propia se quedó en 20 % (azar entre 5 clases) con `val_loss` ≈ 111, y EfficientNetB0 y
ResNet50V2 llegaron a pérdida `NaN`. Los dos transformers entrenaron bien en esa corrida (ViT
99,85 % y DeiT 99,42 % en validación). Las CNN sí entrenaron bien con 1 GPU en Hyperband y en la
validación cruzada.

**Hallazgo de la corrida definitiva:** ViT-B/16 también llegó a `NaN` con 2 GPU, en la época 3
(lote 1.493), y se reentrenó automáticamente con 1 GPU. DeiT-S fue el único modelo que completó
el entrenamiento distribuido.

**Interpretación (hipótesis, no verificada por separado):** la combinación `MirroredStrategy` +
precisión mixta (`mixed_float16`) es numéricamente inestable en esta configuración. Las CNN, cuya
**BatchNormalization** mantiene estadísticas que se sincronizan entre GPU, fallan desde la primera
época. Los transformers (**LayerNormalization**) aguantan más, pero ViT también terminó fallando.
Conclusión práctica: el entrenamiento distribuido acelera (1,66× medido con DeiT-S), pero no es
transparente; hay que vigilar la pérdida y tener un plan de respaldo.

Por eso las CNN se entrenan con 1 GPU y los transformers con 2. Si aun así una pérdida llega
a `NaN`, el script lo detecta (`TerminateOnNaN`) y el modelo se reintenta con 1 GPU.

Con 2 GPU, cada una recibe el lote que eligió Hyperband, así que el lote global se duplica.

La tasa de aprendizaje **no** se escala con el número de GPU (la llamada regla de escalado
lineal la duplicaría). La validación cruzada mostró que EfficientNetB0 y la CNN propia ya son
inestables con su tasa actual: hay folds que caen a 72 % y 78 %. Duplicarla aumentaría ese riesgo.
Además, *early stopping* con `restore_best_weights` conserva la mejor época según validación,
y eso protege de terminar en una época mala.

- **Validación:** solo para *early stopping* y para guardar los mejores pesos.
- **Test:** se evalúa al final, una sola vez por modelo.

In [ ]:
GPUS = {"cnn_propia": 1, "efficientnetb0": 1, "resnet50v2": 1, "vit_b16": 2, "deit_s": 2}

for name in MODEL_NAMES:
    print(f"\n=== {name} ({GPUS[name]} GPU) ===", flush=True)
    codigo, _ = correr("--model", name, "--gpus", str(GPUS[name]))
    if codigo != 0 and GPUS[name] > 1:
        print(f"\n{name} falló con 2 GPU (código {codigo}); se reintenta con 1 GPU.", flush=True)
        codigo, _ = correr("--model", name, "--gpus", "1")
    if codigo != 0:
        print(f"\n{name} falló con 1 GPU; se continúa con el siguiente modelo.")

## 4. Resultados en test

Como en validación cruzada, la exactitud queda muy cerca de 100 % en casi todos, así que la
comparación también mira log-loss, tiempo de entrenamiento, tiempo de inferencia y tamaño.

**Sobre el tamaño:** `size_MB` es el archivo `.keras`, que además de los pesos guarda el estado
del optimizador (necesario solo para seguir entrenando). `pesos_MB` es lo que realmente ocupa el
modelo al usarlo para predecir: 4 bytes por parámetro.

In [ ]:
def metricas(y, probs):
    probs = probs.astype("float64")
    probs = probs / probs.sum(axis=1, keepdims=True)
    pred = probs.argmax(1)
    return {
        "accuracy": accuracy_score(y, pred),
        "f1_macro": f1_score(y, pred, average="macro"),
        "precision_macro": precision_score(y, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y, pred, average="macro", zero_division=0),
        "log_loss": log_loss(y, probs, labels=list(range(len(CLASSES)))),
    }


y_test = np.load(OUT_DIR / "test_y.npy")
filas = []
for name in MODEL_NAMES:
    info_path = OUT_DIR / f"info_{name}.json"
    if not info_path.exists():
        print("Sin resultados:", name)
        continue
    info = json.loads(info_path.read_text())
    if np.isnan(np.load(OUT_DIR / f"test_probs_{name}.npy")).any():
        print("Probabilidades con NaN, se omite:", name)
        continue
    filas.append({**info, **metricas(y_test, np.load(OUT_DIR / f"test_probs_{name}.npy"))})

test_res = pd.DataFrame(filas).set_index("model")
# El archivo .keras también guarda el estado del optimizador (AdamW), que no se usa para predecir;
# el tamaño real del modelo desplegado son sus pesos: 4 bytes por parámetro en float32.
test_res["pesos_MB"] = test_res["params_M"] * 4
test_res.to_csv(OUT_DIR / "test_resultados.csv")
test_res[["accuracy", "f1_macro", "log_loss", "epochs_run", "gpus", "train_min",
          "infer_ms_img", "params_M", "pesos_MB", "size_MB"]].round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, (col, titulo) in zip(axes, [("infer_ms_img", "Inferencia (ms por imagen)"),
                                     ("pesos_MB", "Tamaño de los pesos (MB)"),
                                     ("train_min", "Entrenamiento (min)")]):
    ax.scatter(test_res[col], test_res["accuracy"], s=80)
    for m, r in test_res.iterrows():
        ax.annotate(m, (r[col], r["accuracy"]), textcoords="offset points", xytext=(6, 4))
    ax.set_xlabel(titulo)
    ax.set_ylabel("Accuracy en test")
plt.suptitle("Exactitud frente a costo")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(test_res), figsize=(4.2 * len(test_res), 4.2), squeeze=False)
f1_clase = {}
for ax, name in zip(axes[0], test_res.index):
    pred = np.load(OUT_DIR / f"test_probs_{name}.npy").argmax(1)
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, labels=list(range(len(CLASSES))), display_labels=CLASSES,
        ax=ax, colorbar=False, xticks_rotation=45)
    ax.set_title(name)
    rep = classification_report(y_test, pred, labels=list(range(len(CLASSES))),
                                target_names=CLASSES, output_dict=True, zero_division=0)
    f1_clase[name] = {c: rep[c]["f1-score"] for c in CLASSES}
plt.tight_layout()
plt.show()

f1_clase = pd.DataFrame(f1_clase).T
f1_clase.to_csv(OUT_DIR / "test_f1_por_clase.csv")
f1_clase.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name in test_res.index:
    h = pd.read_csv(OUT_DIR / f"history_{name}.csv")
    axes[0].plot(h["epoch"] + 1, h["val_loss"], marker="o", label=name)
    axes[1].plot(h["epoch"] + 1, h["val_accuracy"], marker="o", label=name)
axes[0].set_yscale("log")
axes[0].set_title("val_loss por época")
axes[1].set_title("val_accuracy por época")
for ax in axes:
    ax.set_xlabel("Época")
    ax.legend()
plt.tight_layout()
plt.show()